<a href="https://colab.research.google.com/github/AlessiaSerra/Clickbait-Detection/blob/main/ClickBaitDetection_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
pip install datasets pandas scikit-learn numpy

In [10]:
#!/usr/bin/env python3
"""
Clickbait Detection via Pure Algorithmic Information Theory
Implementazione KNN con Matrice Normalized Compression Distance (NCD) Precalcolata
"""

import zlib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
from datasets import load_dataset

class PureNCDClassifier:
    """
    Classificatore basato su KNN che utilizza la Normalized Compression Distance (NCD)
    come metrica di distanza precalcolata tra i testi.
    """
    def __init__(self, n_neighbors: int = 3):
        self.knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric='precomputed')
        self.train_texts = []

    @staticmethod
    def _z(text: str) -> float:
        """Calcola la dimensione del testo compresso in byte (Z) usando zlib."""
        if not text:
            return 0.0
        return float(len(zlib.compress(text.encode('utf-8'), level=9)))

    @classmethod
    def _ncd(cls, x: str, y: str) -> float:
        """
        Calcola la Normalized Compression Distance tra due testi x e y.
        Formula: (Z(x+y) - min(Z(x), Z(y))) / max(Z(x), Z(y))
        """
        zx = cls._z(x)
        zy = cls._z(y)
        zxy = cls._z(x + " " + y)

        min_z = min(zx, zy)
        max_z = max(zx, zy)

        if max_z == 0.0:
            return 0.0

        distance = (zxy - min_z) / max_z
        return max(0.0, distance)

    def _compute_distance_matrix(self, X_row: list, X_col: list) -> np.ndarray:
        """
        Genera una matrice di distanze NCD di dimensione (len(X_row), len(X_col)).
        """
        n_rows = len(X_row)
        n_cols = len(X_col)
        dist_matrix = np.zeros((n_rows, n_cols))

        for i in range(n_rows):
            for j in range(n_cols):
                if X_row is X_col and i == j:
                    dist_matrix[i, j] = 0.0
                elif X_row is X_col and j < i:
                    dist_matrix[i, j] = dist_matrix[j, i]
                else:
                    dist_matrix[i, j] = self._ncd(X_row[i], X_col[j])

        return dist_matrix

    def fit(self, X_train: list, y_train: np.ndarray):
        """Calcola la matrice di distanza interna al training set e addestra il KNN."""
        self.train_texts = X_train
        print(f"[*] Calcolo matrice NCD di training ({len(X_train)}x{len(X_train)})...")
        train_dist_matrix = self._compute_distance_matrix(self.train_texts, self.train_texts)
        self.knn.fit(train_dist_matrix, y_train)

    def predict(self, X_test: list) -> np.ndarray:
        """Calcola la matrice di distanza tra test set e training set e genera le predizioni."""
        print(f"[*] Calcolo matrice NCD di test ({len(X_test)}x{len(self.train_texts)})...")
        test_dist_matrix = self._compute_distance_matrix(X_test, self.train_texts)
        return self.knn.predict(test_dist_matrix)


def load_real_clickbait_hf(sample_size: int = 1000) -> pd.DataFrame:
    """
    Scarica un dataset reale da Hugging Face e lo campiona per
    mantenere tempi di esecuzione gestibili per la matrice NCD.
    """
    print("[*] Download del dataset da Hugging Face in corso...")

    # Sostituito victor-upb/clickbait_detection con un dataset pubblico affidabile
    dataset = load_dataset("christinacdl/clickbait_detection_dataset", split="train")
    df = pd.DataFrame(dataset)

    # Rinomina la colonna contenente il testo in 'headline'
    if 'text' in df.columns:
        df = df.rename(columns={'text': 'headline'})
    elif 'title' in df.columns:
        df = df.rename(columns={'title': 'headline'})

    if 'body' not in df.columns:
        df['body'] = ""

    # Sottocampionamento stratificato sicuro per evitare di calcolare una matrice troppo grande
    print(f"[*] Campionamento di {sample_size} articoli totali in modo bilanciato...")
    df_sample = df.groupby('label', group_keys=False).apply(
        lambda x: x.sample(min(len(x), int(sample_size / 2)), random_state=42)
    )

    return df_sample.reset_index(drop=True)


def main():
    # 1. Carica il dataset REALE da Hugging Face
    df = load_real_clickbait_hf(sample_size=400)

    # 2. Prepariamo il testo
    # Dato che la maggior parte dei dataset pubblici ha solo i titoli, NCD verrà calcolato tra i titoli
    X_texts = (df["headline"] + " " + df["body"]).tolist()
    y = df["label"].values

    # 3. Suddivisione in Training e Test
    X_train, X_test, y_train, y_test = train_test_split(X_texts, y, test_size=0.3, random_state=42, stratify=y)

    # 4. Inizializzazione e Addestramento
    classifier = PureNCDClassifier(n_neighbors=5)
    classifier.fit(X_train, y_train)

    # 5. Predizione
    y_pred = classifier.predict(X_test)

    # 6. Risultati
    print("\n" + "=" * 60)
    print(" RISULTATI CLASSIFICAZIONE KNN CON METRICA NCD (DATASET REALE) ")
    print("=" * 60)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["Standard News", "Clickbait"]))


if __name__ == "__main__":
    main()

[*] Download del dataset da Hugging Face in corso...


README.md:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

train.json:   0%|          | 0.00/2.99M [00:00<?, ?B/s]

val.json:   0%|          | 0.00/375k [00:00<?, ?B/s]

test.json:   0%|          | 0.00/373k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30296 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3787 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3787 [00:00<?, ? examples/s]

[*] Campionamento di 400 articoli totali in modo bilanciato...
[*] Calcolo matrice NCD di training (280x280)...


/tmp/ipykernel_1177/395652020.py:105: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby('label', group_keys=False).apply(


[*] Calcolo matrice NCD di test (120x280)...

 RISULTATI CLASSIFICAZIONE KNN CON METRICA NCD (DATASET REALE) 
Accuracy: 0.8167

Classification Report:
               precision    recall  f1-score   support

Standard News       0.75      0.95      0.84        60
    Clickbait       0.93      0.68      0.79        60

     accuracy                           0.82       120
    macro avg       0.84      0.82      0.81       120
 weighted avg       0.84      0.82      0.81       120

